# Preprocessing for the LFQ_diaPASEF dataset

In [1]:
import pandas as pd
import numpy as np
import re
from src.column_spec import ColumnSpec
from src.lfq_diaPASED_pretreatment import *
from src.transformations import *

from src.QC import *
from src.plotting_functions import *
pio.renderers.default = "png"

## Renaming and ordering columns

In [2]:
CELL_LINES = ["WT", "EGFRT693A", "BRAFS151A1", "SOS1S1178A", "SHOC2T71A", "BRAFS151A2", "GAB1Y259A", "RPS6KA3S375A"]
TIME_POINTS = ["full", "starve", "2", "5", "10", "15", "20", "30", "90"]
REPLICATES = ["r1", "r2", "r3"]
CONDITION = "EGF" #"_EGF_"
DATA_TYPE = "raw:abs"

# Cell lines naming dictionary
labes_dic = {}
c = 1
for cell in CELL_LINES:
    for tp in TIME_POINTS:
        for rep in REPLICATES:
            labes_dic[c] = cell + "_" + DATA_TYPE + "_" + CONDITION + "_" + tp + "_" + rep
            c += 1

# Control channels naming
MIX_LABELS = {"mix":  "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r1",
              "mixb": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r2",
              "mixc": "MIX_" + DATA_TYPE + "_" + CONDITION + "_starve_r3",}

# Importing dataset
df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/dia-quant-output/abundance_multi-site_MS2quant_None.tsv", sep="\t")

# Annotation columns -> project names
regular_rename = {
    "Index": "peptide_index",
    "Gene": "protein_name",
    "ProteinID": "protein_Id",
    "Peptide": "peptide_seq",
}

# Run columns -> naming convention
rename_cols = {}
unmatched = []
for col in df.columns:
    if not col.startswith("E:"):
        continue
    token = re.search(r"_([^_]+)\.d$", col) #sample_number
    if token is None:
        unmatched.append(col)
        continue
    sample = token.group(1)
    if sample.isdigit() and int(sample) in labes_dic:
        rename_cols[col] = labes_dic[int(sample)]
    elif sample in MIX_LABELS:
        rename_cols[col] = MIX_LABELS[sample]
    else:
        unmatched.append(col)


df = df.rename(columns=rename_cols)
df = df.rename(columns=regular_rename)

# Standard site column
df["site"] = df["peptide_index"] + "~" + df["peptide_seq"]  # This peptide sequence is not the correct one (missing the localization in small letters)

# --- Column ordering, computed AFTER every rename ------------------------------------------
sample_cols = [name for name in labes_dic.values() if name in df.columns]
mix_cols = [name for name in MIX_LABELS.values() if name in df.columns]
run_cols = set(sample_cols) | set(mix_cols)
other_cols = [col for col in df.columns if col not in run_cols]

df = df[other_cols + sample_cols + mix_cols]

# --- Report --------------------------------------------------------------------------------
missing = [name for name in list(labes_dic.values()) + list(MIX_LABELS.values())
           if name not in df.columns]
print(f"renamed {len(sample_cols)} experimental runs + {len(mix_cols)} mix controls; "
      f"{len(other_cols)} annotation columns")
if missing:
    print(f"  expected but absent ({len(missing)}): {missing}")
if unmatched:
    print(f"  unrecognised run columns ({len(unmatched)}): {unmatched}")
assert len(other_cols) + len(sample_cols) + len(mix_cols) == df.shape[1], "columns lost while reordering"
assert not df.columns.duplicated().any(), "duplicate column names after renaming"

print(df.shape)
df.head(5)

renamed 216 experimental runs + 3 mix controls; 21 annotation columns
(71087, 240)


,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,RPS6KA3S375A_raw:abs_EGF_20_r3,RPS6KA3S375A_raw:abs_EGF_30_r1,RPS6KA3S375A_raw:abs_EGF_30_r2,RPS6KA3S375A_raw:abs_EGF_30_r3,RPS6KA3S375A_raw:abs_EGF_90_r1,RPS6KA3S375A_raw:abs_EGF_90_r2,RPS6KA3S375A_raw:abs_EGF_90_r3,MIX_raw:abs_EGF_starve_r1,MIX_raw:abs_EGF_starve_r2,MIX_raw:abs_EGF_starve_r3
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,1714.0734,NaN,3342.1428,2313.1060,NaN,NaN,1156.0444,2429.0979,3397.1465,2826.1235
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,491.0169,1076.0327,463.0141,NaN,1074.0334,1567.0479,392.0133,NaN,942.0345,514.0154
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,NaN,NaN,NaN,1756.0579,NaN,NaN,1320.0422,729.0223,800.0243,NaN
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,801.0384,NaN,NaN,NaN,NaN,NaN,723.0341,NaN,1333.0662,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,367.0161,560.0273,784.0388,NaN,380.0154,545.0200,NaN,734.0343,497.0189,449.0199


### Incorprorate proper "site" identifier column

In [3]:
df = add_site_identificator(df,
                            mod_col = "Best Precursor for Quant",
                            peptide_index_col= "peptide_index")
print(df.shape)
df.head(5)

(71087, 240)


,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,RPS6KA3S375A_raw:abs_EGF_20_r3,RPS6KA3S375A_raw:abs_EGF_30_r1,RPS6KA3S375A_raw:abs_EGF_30_r2,RPS6KA3S375A_raw:abs_EGF_30_r3,RPS6KA3S375A_raw:abs_EGF_90_r1,RPS6KA3S375A_raw:abs_EGF_90_r2,RPS6KA3S375A_raw:abs_EGF_90_r3,MIX_raw:abs_EGF_starve_r1,MIX_raw:abs_EGF_starve_r2,MIX_raw:abs_EGF_starve_r3
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,1714.0734,NaN,3342.1428,2313.1060,NaN,NaN,1156.0444,2429.0979,3397.1465,2826.1235
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,491.0169,1076.0327,463.0141,NaN,1074.0334,1567.0479,392.0133,NaN,942.0345,514.0154
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,NaN,NaN,NaN,1756.0579,NaN,NaN,1320.0422,729.0223,800.0243,NaN
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,801.0384,NaN,NaN,NaN,NaN,NaN,723.0341,NaN,1333.0662,NaN
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,367.0161,560.0273,784.0388,NaN,380.0154,545.0200,NaN,734.0343,497.0189,449.0199


### Saving dataset with only raw abundances

In [5]:
# df.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_preprocessed.tsv", sep= "\t", index=False)

# Data transformation

`run_diapasef_transformations` (`src/transformations.py`, *diaPASEF (LFQ) transformations* section)
applies the chain below to every cell line in one pass:

| step | columns produced |
|---|---|
| 1 | `raw:mean`, `raw:median`, `raw:sd`, `raw:cv` — zeros treated as missing, so a non-detection does not drag the mean down |
| 2 | `log2:abs` — per replicate, zeros → NaN (`log2(0)` is undefined, and a zero here means *not detected*) |
| 3 | `log2:mean`, `log2:median`, `log2:sd` — statistics *of the log2 values*, i.e. the log2 geometric mean |
| 4 | `log2:FC` — `log2:mean(t) − log2:mean(starve)`, per cell line per condition |
| 5 | `log2:scaled` — `log2:FC / max(|log2:FC|)`, the maximum taken per cell line **across its conditions**, so relative amplitude between conditions survives |
| 6 | `log2:zscore` — `(log2:FC − mean_t) / sd_t`, per cell line **per condition**, i.e. shape with the amplitude removed |

**No starve, no fold change.** A site with no abundance detected in starve has no reference, so the
site is skipped and all of its `log2:FC` columns for that cell line stay **NaN** — not 0, which would
read as "no change" everywhere downstream. This is decided per site *per cell line*: a site can have
a usable FC in WT and none in a mutant. The counts are printed so the loss is visible.
`log2:FC_{treatment}_starve` is computed and is identically 0 — the rest of the project relies on
that structural zero.

**Why a separate section rather than `run_all_transformations`.** Three things differ for this dataset:
`_sort_timepoints()` sorts against a fixed `_TP_ORDER` that has no 20 or 30 min, so the diaPASEF grid
would come out as `… 15, 90, 20, 30`; all 8 cell lines are processed together with each column assigned
by its exact first field (a name that prefixes another cannot pull in both); and the ~1500 new columns
are attached with one `concat` per block instead of column-by-column insertion. The TMT functions are
untouched.

**The normalisation basis, and why it is not the TMT one.** Steps 5 and 6 are two *alternative*
representations of the same `log2:FC` values — neither reads the other, both are written, the analysis
downstream picks one. They differ from `compute_scaled_fc()` / `compute_zscore_fc()` in a single
deliberate way: which timepoints define the normalisation.

- `full` is excluded from both bases (`exclude_from_scale`, `exclude_from_zscore_basis`). Full media is
  a different media state, not a response to EGF, and its |FC| vs starve is routinely the largest value
  in the row — as the scaling denominator it squashes the actual response, and in the z-score basis it
  was measured on hme1_2 to carry ~26% of the clustering variance (`clustering_method_decision.md` §1).
- `starve` is additionally excluded from the z-score basis: it is identically 0 in FC space, so it
  contributes a constant rather than information.

The columns are still written for **every** timepoint — only the basis changes. So `log2:scaled` at
`full` may exceed 1, and `log2:zscore` at `starve` reads as how far the baseline sits below the mean
response, in SDs. `min_zscore_timepoints=3` blanks sites whose basis has fewer than three measured
points, where a sd is not a shape. Passing `exclude_from_scale=()`, `exclude_from_zscore_basis=()` and
`min_zscore_timepoints=1` reproduces the TMT functions exactly (verified).

No p-values or FDR: differential statistics are computed downstream in R/limma.

In [4]:
# All 8 cell lines in one pass. min_reps=1 reports whatever was detected; raise it to 2 to blank statistics computed from a single replicate (sd is NaN for a single replicate either way).
df_transformed = run_diapasef_transformations(df,
                                              cell_lines=CELL_LINES,
                                              conditions=["_EGF_"],
                                              data_type= DATA_TYPE,
                                              min_reps=1,
                                              reference="starve",
                                              verbose=True,)
print(df_transformed.shape)
df_transformed.head()

Found 8 cell lines, 72 (cell line, condition, timepoint) groups.
Sites without a 'starve' reference (log2:FC left as NaN):
  WT             EGF          20837 / 71087 (29.3%)
  EGFRT693A      EGF          22796 / 71087 (32.1%)
  BRAFS151A1     EGF          21846 / 71087 (30.7%)
  SOS1S1178A     EGF          22546 / 71087 (31.7%)
  SHOC2T71A      EGF          27620 / 71087 (38.9%)
  BRAFS151A2     EGF          22259 / 71087 (31.3%)
  GAB1Y259A      EGF          20523 / 71087 (28.9%)
  RPS6KA3S375A   EGF          23672 / 71087 (33.3%)
log2:scaled — scale factor from timepoints excluding ('full',):
  WT               49221 / 71087 sites scaled
  EGFRT693A        47516 / 71087 sites scaled
  BRAFS151A1       48148 / 71087 sites scaled
  SOS1S1178A       48068 / 71087 sites scaled
  SHOC2T71A        43144 / 71087 sites scaled
  BRAFS151A2       48210 / 71087 sites scaled
  GAB1Y259A        50006 / 71087 sites scaled
  RPS6KA3S375A     46881 / 71087 sites scaled
log2:zscore — basis excludes 

,peptide_index,protein_name,protein_Id,peptide_seq,SequenceWindow,Start,End,Peptide Length,Probability,Protein,...,GAB1Y259A_log2:zscore_EGF_90,RPS6KA3S375A_log2:zscore_EGF_full,RPS6KA3S375A_log2:zscore_EGF_starve,RPS6KA3S375A_log2:zscore_EGF_2,RPS6KA3S375A_log2:zscore_EGF_5,RPS6KA3S375A_log2:zscore_EGF_10,RPS6KA3S375A_log2:zscore_EGF_15,RPS6KA3S375A_log2:zscore_EGF_20,RPS6KA3S375A_log2:zscore_EGF_30,RPS6KA3S375A_log2:zscore_EGF_90
0,P16333_147_176_1_1_S166,NCK1,P16333,GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK,KCSDGWWR.GSYNGQVGWFPSNYVTEEGDSPLGDHVGSLSEK.LAA...,146,178,33,1.0,P16333,...,-2.254722,1.138739,1.921288,-0.170591,0.979981,0.802300,0.195404,0.354294,0.116667,-2.278055
1,Q9Y618_1595_1611_1_0,NCOR2,Q9Y618,EIAKSPHSTVPEHHPHPISPYEHLLR;SPHSTVPEHHPHPISPYEHLLR,RKLTSTPR.EIAKSPHSTVPEHHPHPISPYEHLLR.GVSGVDLY,1591,1616,26,1.0,Q9Y618,...,NaN,1.683852,0.079272,-0.399912,0.316632,0.491135,2.056818,-0.817499,-1.172303,-0.474870
2,Q9H2G2_565_571_2_2_S565T569,SLK,Q9H2G2,VDEDSAEDTQSNDGK;VDEDSAEDTQSNDGKEVVEVGQK,EAADVAQK.VDEDSAEDTQSNDGKEVVEVGQK.LINKPMVG,561,583,23,1.0,Q9H2G2,...,-0.075238,-1.783092,-0.555592,NaN,0.785452,-2.012114,-0.452873,0.504968,0.891071,0.283496
3,O15085_1452_1469_1_1_S1466,ARHGEF11,O15085,SLGGESSGGTTPVGSFHTEAAR,LAHRELLK.SLGGESSGGTTPVGSFHTEAAR.WTDGSLSP,1452,1473,22,1.0,O15085,...,-1.373734,2.564218,0.482151,1.693951,NaN,-0.180434,-1.307363,0.325598,NaN,-0.531753
4,P08670_412_420_1_1_S419,VIM,P08670,ISLPLPNFSSLNLR,LLEGEESR.ISLPLPNFSSLNLR.ETNLDSLP,411,424,14,1.0,P08670,...,-1.134151,1.149598,0.152700,-0.957564,0.675625,NaN,0.341861,-1.630081,1.341798,0.228361


### Checking the output how many sites are there for which a FC could be computed

If FC was computed this means that the site was detected at starvation and another time point (at least one)

In [7]:
site_row = df_transformed.index[0]

profile = {}
for dtype in ["raw:mean", "raw:median", "raw:sd", "raw:cv", "log2:mean", "log2:median", "log2:sd", "log2:FC", "log2:scaled", "log2:zscore"]:
    cols = ColumnSpec.select(df_transformed,
                             cell_lines=["WT"],
                             data_type=dtype,
                             conditions=["_" + CONDITION + "_"],)
    profile[dtype] = pd.Series(df_transformed.loc[site_row, cols].values, index=[c.split("_")[-1] for c in cols],)

print(df_transformed.loc[site_row, "site"])
pd.DataFrame(profile).T.round(3)

P16333_147_176_1_1_S166~GSYNGQVGWFPSNYVTEEGDsPLGDHVGSLSEK


,full,starve,2,5,10,15,20,30,90
raw:mean,4453.1897,3954.173733,3094.140933,2910.792467,2307.102733,1242.0534,NaN,2350.0983,1145.04675
raw:median,4453.1897,2937.1294,2506.1072,3531.158,2294.1125,1369.0552,NaN,2350.0983,1145.04675
raw:sd,466.697688,2225.327224,1717.33004,1157.742556,1252.60477,360.694145,NaN,956.046411,530.357168
raw:cv,10.480077,56.277932,55.502644,39.774136,54.293411,29.040148,NaN,40.681124,46.317512
log2:mean,12.116651,11.809355,11.452873,11.41045,11.005047,10.232159,NaN,11.136201,10.079342
log2:median,12.116651,11.520191,11.291232,11.785926,11.16372,10.418965,NaN,11.136201,10.079342
log2:sd,0.151473,0.756344,0.774895,0.683777,0.885173,0.462292,NaN,0.603947,0.693779
log2:FC,0.307296,0.0,-0.356482,-0.398905,-0.804308,-1.577195,NaN,-0.673154,-1.730013
log2:scaled,0.177627,0.0,-0.206057,-0.230579,-0.464914,-0.911667,NaN,-0.389104,-1.0
log2:zscore,2.277809,1.70903,1.049212,0.97069,0.220324,-1.210226,NaN,0.463078,-1.493078


In [9]:
# How many sites ended up with a usable fold-change profile, per cell line — the complement of the "no starve reference" report printed above.
for cell in CELL_LINES:
    fc_cols = ColumnSpec.select(df_transformed,
                                cell_lines=[cell],
                                data_type="log2:FC",
                                conditions=["_" + CONDITION + "_"],)
    n_any = int(df_transformed[fc_cols].notna().any(axis=1).sum())
    n_all = int(df_transformed[fc_cols].notna().all(axis=1).sum())
    print(f"{cell:<14} FC at >=1 timepoint: {n_any:>6} | at every timepoint: {n_all:>6} "
          f"| of {len(df_transformed)}")

WT             FC at >=1 timepoint:  50250 | at every timepoint:  23175 | of 71087
EGFRT693A      FC at >=1 timepoint:  48291 | at every timepoint:  22873 | of 71087
BRAFS151A1     FC at >=1 timepoint:  49241 | at every timepoint:  23317 | of 71087
SOS1S1178A     FC at >=1 timepoint:  48541 | at every timepoint:  24158 | of 71087
SHOC2T71A      FC at >=1 timepoint:  43467 | at every timepoint:  23555 | of 71087
BRAFS151A2     FC at >=1 timepoint:  48828 | at every timepoint:  24676 | of 71087
GAB1Y259A      FC at >=1 timepoint:  50564 | at every timepoint:  25092 | of 71087
RPS6KA3S375A   FC at >=1 timepoint:  47415 | at every timepoint:  23348 | of 71087


In [19]:
# Save. Never overwrite an existing file — the name carries the transformation step.
# df_transformed.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_nolimma.tsv",sep="\t", index=False)
# df_transformed.head(200).to_csv("../../data/20260818_peptide_MS2quant_None_transformed_nolimma_sample.tsv",sep="\t", index=False)
df_transformed.head(5)

# Import Limma processed data and merge it

In [20]:
limma_pvalues_df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv", sep="\t")
limma_pvalues_df.head(5)

,site,WT_log2:limmaFC_EGF_full,WT_log2:limmaFC_EGF_2,WT_log2:limmaFC_EGF_5,WT_log2:limmaFC_EGF_10,WT_log2:limmaFC_EGF_15,WT_log2:limmaFC_EGF_20,WT_log2:limmaFC_EGF_30,WT_log2:limmaFC_EGF_90,WT_log2:pvalue_EGF_full,...,RPS6KA3S375A_log2:adjustedFDR_EGF_15,RPS6KA3S375A_log2:adjustedFDR_EGF_20,RPS6KA3S375A_log2:adjustedFDR_EGF_30,RPS6KA3S375A_log2:adjustedFDR_EGF_90,RPS6KA3S375A_log2:Fpvalue_EGF_omnibus,RPS6KA3S375A_log2:Fpvalue_ALL_omnibus,RPS6KA3S375A_log2:FFDR_EGF_omnibus,RPS6KA3S375A_log2:FFDR_ALL_omnibus,RPS6KA3S375A_log2:adjustedFFDR_EGF_omnibus,RPS6KA3S375A_log2:adjustedFFDR_ALL_omnibus
0,A0A0B4J2A2_93_116_1_0~HtGSGILSMANAGPNTNGSQFFIC...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A0A0B4J2A2_93_119_1_0~HTGSGILSMANAGPNTNGSQFFIC...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000076,0.000007,1.940341e-07,0.002318,0.880199,0.880199,0.999992,0.999992,0.000004,0.000004
2,A0A3B3IU46_36_45_1_1_S36~RPPEsPPIVEEWNSR,-0.649117,-0.474191,-0.501739,-1.750213,-0.722565,-0.704210,-0.331648,-1.407394,0.271197,...,0.000076,0.000007,1.940341e-07,0.002318,0.969307,0.969307,0.999992,0.999992,0.000004,0.000004
3,A0AVK6_346_366_1_0~WTGPEISPNtSGSSPVIHFTPSDLEVR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0AVT1_951_953_1_1_S951~NGIsFTIWDR,0.221409,-0.773648,-0.147294,-1.094043,-0.596615,-0.582538,-0.862279,-1.168520,0.719146,...,0.000076,0.000007,1.940341e-07,0.002318,0.800405,0.800405,0.999992,0.999992,0.000004,0.000004


In [22]:
hme1_diapasef_limma = merge_limma_results(df_transformed,
                                          limma_path = "../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv",
                                          key="site")

  merged 304 limma columns from ../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_limma_pvalues.tsv
  27096 / 71087 sites carry statistics (43991 not tested by limma)


In [23]:
# hme1_diapasef_limma.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_limma.tsv", sep="\t", index=False)

# Merge Phosphositeplus data

In [24]:
functional_score_df = pd.read_csv("../../External_Data/Metadata/PhosphoSitePlus.tsv", sep="\t")
regulatory_sites = pd.read_csv("../../External_Data/Metadata/Phosphosite/Regulatory_sites.tsv", sep="\t")

print(f"functional_score_df columns: {functional_score_df.columns}")
print(f"regulatory_sites columns: {regulatory_sites.columns}")

functional_score_df columns: Index(['protein_Id', 'protein_name', 'prot_seq_position', 'aa', 'site',
       'functional_score', 'ms_lit', 'ERK_motif', 'ERK_ext_motif'],
      dtype='object')
regulatory_sites columns: Index(['GENE', 'protein_name', 'information', 'protein_Id', 'GENE_ID',
       'HU_CHR_LOC', 'ORGANISM', 'MOD_RSD', 'SITE_GRP_ID', 'SITE_+/-7_AA',
       'DOMAIN', 'ON_FUNCTION', 'ON_PROCESS', 'ON_PROT_INTERACT',
       'ON_OTHER_INTERACT', 'PMIDs', 'LT_LIT', 'MS_LIT', 'MS_CST', 'NOTES'],
      dtype='object')


In [25]:
df_with_extra_info = merge_functional_score(df = hme1_diapasef_limma,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position")
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = functional_score_df,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ERK_motif"])
df_with_extra_info = merge_phosphoplus_info(df = df_with_extra_info,
                                            phosphosite_df = regulatory_sites,
                                            ph_residue ="aa",
                                            ph_position = "prot_seq_position",
                                            adding_info = ["ORGANISM", "ON_FUNCTION", "ON_PROCESS", "ON_PROT_INTERACT", 'ON_OTHER_INTERACT'],
                                            regulatory_sites= True)

In [28]:
#df_with_extra_info.to_csv("../../Experiment/hme1_diaPASEF/Data/Processed/20260818_peptide_MS2quant_None_transformed_limma_phPlus.tsv", sep="\t", index=False)
